In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import statsmodels.api as sm
import ta
import os

import refinitiv.dataplatform as rdp
import lseg.data as ld

data_folder = os.getcwd() + '/'
data_folder
# https://github.com/LSEG-API-Samples/Example.RDPLibrary.Python

/opt/anaconda3/lib/python3.13/site-packages/refinitiv/dataplatform/__init__.py:34: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


'/Users/gunassavsopee/Gun/Python/TEST_MVP/36****/MACRO_DATA/'

In [3]:
# GET DATA FORM COUNTRY INDEX
country = {'EK':'EZU','AU':'EWA','JP':'EWJ','UK':'EWU','US':'IVV','CH':'MCHI','IN':'INDA','VI':'VNM','TH':'THD'}
xls = pd.ExcelFile('MACRO_by_cat_sum_no_head.xlsx')

for i in country:
    df = pd.DataFrame()
    for sheet in xls.sheet_names:
        macro = pd.read_excel(xls, sheet_name=sheet)
        macro.rename(columns={'Name':'Date'},inplace=True)
        macro = macro.set_index('Date')
        macro_country = macro[[col for col in macro.columns if col[:2] == i]]
        try:
            macro_country = macro_country.iloc[:, [1, 2]]
            macro_country.columns = [sheet+'_MOM',sheet+'_YOY']
            df = df.merge(macro_country,left_index=True,right_index=True,how='outer')
            print(i +" "+ sheet)
        except:
            print(' No data for ' + i +" "+ sheet)

    end = df.index.max() + pd.offsets.MonthEnd(0)
    new_index = pd.date_range(start=df.index.min(),end=end,freq='D')
    df = df.reindex(new_index).ffill()
    
    file_name = country[i] + '_Macro_M'
    df.to_excel(file_name + '.xlsx')
    print(file_name +' Done')
    print('_'*30)
    print('_'*30)

EK EXP
EK IMP
EK CA
EK CPI
EK CORE_CPI
EK PPI
EK UNEM
 No data for EK RS
EK IP
EZU_Macro_M Done
______________________________
______________________________
AU EXP
AU IMP
AU CA
AU CPI
AU CORE_CPI
AU PPI
AU UNEM
AU RS
AU IP
EWA_Macro_M Done
______________________________
______________________________
JP EXP
JP IMP
JP CA
JP CPI
JP CORE_CPI
JP PPI
JP UNEM
JP RS
JP IP
EWJ_Macro_M Done
______________________________
______________________________
UK EXP
UK IMP
UK CA
UK CPI
UK CORE_CPI
UK PPI
UK UNEM
UK RS
UK IP
EWU_Macro_M Done
______________________________
______________________________
US EXP
US IMP
US CA
US CPI
US CORE_CPI
US PPI
US UNEM
US RS
US IP
IVV_Macro_M Done
______________________________
______________________________
CH EXP
CH IMP
CH CA
CH CPI
CH CORE_CPI
 No data for CH PPI
CH UNEM
CH RS
 No data for CH IP
MCHI_Macro_M Done
______________________________
______________________________
IN EXP
IN IMP
IN CA
IN CPI
 No data for IN CORE_CPI
IN PPI
 No data for IN UNEM
 No data f

In [13]:
#  CHECK COLUMN NAME
stocks = ['EZU','EWA','EWJ','EWU','IVV','MCHI','INDA','VNM','THD']
df = pd.DataFrame(index=range(1, 20))
for stock in stocks:
    macro = pd.read_excel(stock+'_Macro.xlsx')
    macro.rename(columns={'Name':'Date'},inplace=True)
    macro = macro.set_index('Date')
    df[stock] = pd.Series(macro.columns)

df

,EZU,EWA,EWJ,EWU,IVV,MCHI,INDA,VNM,THD
1,GBY2Y,GBY2Y,GBY2Y,GBY2Y,GBY2Y,GBY2Y,GBY2Y,GBY2Y,GBY2Y
2,GBY5Y,GBY5Y,GBY5Y,GBY5Y,GBY5Y,GBY5Y,GBY5Y,GBY5Y,GBY5Y
3,GBY10Y,GBY10Y,GBY10Y,GBY10Y,GBY10Y,GBY10Y,GBY10Y,GBY10Y,GBY10Y
4,GBY10-2Y,GBY10-2Y,GBY10-2Y,GBY10-2Y,GBY10-2Y,GBY10-2Y,GBY10-2Y,GBY10-2Y,GBY10-2Y
5,gold_M_diff,gold_M_diff,gold_M_diff,gold_M_diff,gold_M_diff,gold_M_diff,gold_M_diff,gold_M_diff,gold_M_diff
6,BDI_M_diff,BDI_M_diff,BDI_M_diff,BDI_M_diff,BDI_M_diff,BDI_M_diff,BDI_M_diff,BDI_M_diff,BDI_M_diff
7,BRT_M_diff,BRT_M_diff,BRT_M_diff,BRT_M_diff,BRT_M_diff,BRT_M_diff,BRT_M_diff,BRT_M_diff,BRT_M_diff
8,EUR/USD_M_diff,EUR/USD_M_diff,JPY/USD_M_diff,GBP/USD_M_diff,USD_M_diff,CNY/USD_M_diff,INR/USD_M_diff,VND/USD_M_diff,THB/USD_M_diff
9,CPI_MOM,CPI_MOM,CPI_MOM,CPI_MOM,CPI_MOM,CPI_MOM,CPI_MOM,CPI_MOM,CPI_MOM
10,PPI_MOM,PPI_MOM,PPI_MOM,PPI_MOM,PPI_MOM,EXP_MOM,PPI_MOM,EXP_MOM,PPI_MOM


In [4]:
def cleandata(df):
    df = df.astype(float)
    # df[col_name]= np.log(df[col_name]/df[col_name].shift(1))
    df = df.asfreq('d').ffill()  # Fill missing values
    df = df.dropna()
    return df

In [8]:
# Merge daily and monthly data
stocks = ['EZU','EWA','EWJ','EWU','IVV','MCHI','INDA','VNM','THD']
col_m = ['CPI_MOM','PPI_MOM','EXP_YOY','IMP_YOY','CA_YOY','UNEM_MOM','IP_MOM','RS_MOM']
for stock in stocks:
    macro_d = pd.read_excel(stock+'_Macro_D.xlsx')
    macro_d.rename(columns={'Unnamed: 0':'Date'},inplace=True)
    macro_d = macro_d.set_index('Date')
    
    macro_m = pd.read_excel(stock+'_Macro_M.xlsx')
    macro_m.rename(columns={'Unnamed: 0':'Date'},inplace=True)
    macro_m = macro_m.set_index('Date')

    macro_m_data = pd.DataFrame()
    for col in col_m:
        try:
            macro_m_data[col] = macro_m[col]
        except:
            print(f'No DATA for {col}')
    
    macro = macro_d.merge(macro_m_data,left_index=True,right_index=True,how='outer')
    macro = macro.dropna()
        
    file_name = stock + '_Macro'
    macro.to_excel(file_name + '.xlsx')
    print(file_name+' DONE')
    
    if stock == 'IVV':
        macro.to_excel('AGG_Macro.xlsx')
        print('AGG_Macro'+' DONE')


No DATA for RS_MOM
EZU_Macro DONE
EWA_Macro DONE
EWJ_Macro DONE
EWU_Macro DONE
IVV_Macro DONE
AGG_Macro DONE
No DATA for PPI_MOM
No DATA for UNEM_MOM
No DATA for IP_MOM
MCHI_Macro DONE
No DATA for UNEM_MOM
No DATA for RS_MOM
INDA_Macro DONE
No DATA for PPI_MOM
No DATA for UNEM_MOM
VNM_Macro DONE
THD_Macro DONE


In [10]:
x = pd.read_excel('IVV_Macro.xlsx')
x.rename(columns={'Unnamed: 0':'Date'},inplace=True)
x = x.set_index('Date')
x

,VIX,GBY2Y,GBY5Y,GBY10Y,GBY10-2Y,gold_D_diff,gold_M_diff,BDI_D_diff,BDI_M_diff,BRT_D_diff,...,USD_D_diff,USD_M_diff,CPI_MOM,PPI_MOM,EXP_YOY,IMP_YOY,CA_YOY,UNEM_MOM,IP_MOM,RS_MOM
Date,,,,,,,,,,,,,,,,,,,,,
2010-02-03,21.60,0.883,2.407,3.707,2.824,-0.497669,-0.959479,-0.668896,-14.872611,2.458356,...,0.513532,2.488992,0.065,1.016,17.883,11.690,-0.119,-0.1,1.090,0.105
2010-02-04,26.08,0.807,2.303,3.610,2.803,-4.163475,-4.968500,0.448934,-17.889908,-5.926314,...,0.737698,3.005630,0.065,1.016,17.883,11.690,-0.119,-0.1,1.090,0.105
2010-02-05,26.11,0.771,2.240,3.571,2.800,0.188067,-6.449205,1.117318,-16.692237,-1.658537,...,0.321435,3.488717,0.065,1.016,17.883,11.690,-0.119,-0.1,1.090,0.105
2010-02-06,26.11,0.771,2.240,3.571,2.800,0.000000,-5.903912,0.000000,-13.782153,0.000000,...,0.000000,2.851098,0.065,1.016,17.883,11.690,-0.119,-0.1,1.090,0.105
2010-02-07,26.11,0.771,2.240,3.571,2.800,0.000000,-6.259898,0.000000,-13.535032,0.000000,...,0.000000,3.517432,0.065,1.016,17.883,11.690,-0.119,-0.1,1.090,0.105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-26,13.60,3.483,3.696,4.134,0.651,1.182909,8.837114,0.000000,-21.824240,0.000000,...,0.112319,-1.498940,0.298,-0.064,7.564,-1.237,16.200,0.1,0.525,0.000
2025-12-27,13.60,3.483,3.696,4.134,0.651,0.000000,9.021013,0.000000,-24.314516,0.000000,...,0.000000,-1.486074,0.298,-0.064,7.564,-1.237,16.200,0.1,0.525,0.000
2025-12-28,13.60,3.483,3.696,4.134,0.651,0.000000,7.138425,0.000000,-26.679688,0.000000,...,0.000000,-1.454388,0.298,-0.064,7.564,-1.237,16.200,0.1,0.525,0.000


In [42]:
# file_name = stock + '_Macro'
# macro.to_excel('/Users/gunassavsopee/Gun/Python/MACRO_DATA/'+ file_name + '.xlsx')